# PN-TDNN-DPD Training: CWGAN-GP with QAT

**Version:** 2.0 (Production)  
**Architecture:** Phase-Normalized TDNN (24-dim, M=3, 1,362 params)  
**Training:** CWGAN-GP + Spectral Loss + ILA  

## Key Design Decisions

| Component | Specification | Reason |
|-----------|--------------|--------|
| Input dim | 24 | 6 features × 4 taps (M=3) |
| Memory depth | M=3 | Captures >95% GaN PA memory |
| Feature extraction | Phase-normalized | Decouples amplitude/phase learning |
| Output activation | Linear (no Tanh) | Phase denorm bounds output naturally |
| Training | ILA | No PA model in loop |
| QAT | Q1.15 weights, Q8.8 activations | After epoch 300 |

In [ ]:
# Cell 1: Environment Setup
import os
import sys
from pathlib import Path

# Check if running in Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Clone repo if needed
    REPO_PATH = Path('/content/6g-pa-gan-dpd')
    if not REPO_PATH.exists():
        !git clone https://github.com/your-username/6g-pa-gan-dpd.git {REPO_PATH}
    
    os.chdir(REPO_PATH)
    sys.path.insert(0, str(REPO_PATH))
else:
    # Local development
    REPO_PATH = Path('.').resolve()
    sys.path.insert(0, str(REPO_PATH))

print(f"Working directory: {os.getcwd()}")
print(f"Python path includes: {REPO_PATH}")

In [ ]:
# Cell 2: Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from datetime import datetime
import yaml

# Import from project
from models.pn_tdnn_generator import (
    PNTDNNGenerator,
    PhaseNormalizedFeatureExtraction,
    Discriminator,
    create_pn_tdnn_generator,
    create_discriminator
)
from utils.spectral_loss import SpectralLoss

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 3: Configuration (matches train.py and ARCHITECTURE.md)
config = {
    # Model architecture (FROZEN - matches ARCHITECTURE.md)
    'model': {
        'generator': {
            'memory_depth': 3,        # M=3 (24-dim input)
            'hidden_dims': [32, 16],  # FC layers
            'leaky_slope': 0.2,       # LeakyReLU
        },
        'discriminator': {
            'input_dim': 4,           # 2 (output) + 2 (condition)
            'hidden_dims': [64, 32],  # FC layers
            'leaky_slope': 0.2,
        }
    },
    
    # Training hyperparameters
    'training': {
        'epochs': 500,
        'batch_size': 256,
        'n_critic': 5,                # WGAN-GP: train D 5x per G update
        'gp_weight': 10.0,            # Gradient penalty weight
        
        'optimizer': {
            'lr_generator': 1e-4,
            'lr_discriminator': 1e-4,
            'betas': [0.0, 0.9],      # WGAN-GP: no momentum
            'weight_decay': 1e-5,
        },
        
        'scheduler': {
            'type': 'cosine',
            'min_lr': 1e-6,
        },
        
        'loss': {
            'adversarial': 1.0,       # Wasserstein loss weight
            'reconstruction_l1': 50.0, # L1 loss weight
            'spectral': 10.0,         # Spectral loss weight
        },
    },
    
    # QAT configuration
    'qat': {
        'enabled': True,
        'start_epoch': 300,           # Enable QAT after float32 pre-training
        'weight_bits': 16,            # Q1.15
        'activation_bits': 16,        # Q8.8
    },
    
    # System parameters
    'system': {
        'sample_rate': 250e6,         # 250 MSps
    },
    
    # Logging
    'logging': {
        'log_interval': 100,
        'checkpoint_interval': 25,
    }
}

print("Configuration loaded:")
print(f"  Memory depth: M={config['model']['generator']['memory_depth']}")
print(f"  Input dim: {6 * (config['model']['generator']['memory_depth'] + 1)}")
print(f"  Hidden dims: {config['model']['generator']['hidden_dims']}")
print(f"  Epochs: {config['training']['epochs']}")
print(f"  QAT starts at epoch: {config['qat']['start_epoch']}")

In [ ]:
# Cell 4: Data Loading (matches train.py)
def load_measured_data(data_dir: Path, split: str = 'train'):
    """
    Load measured PA input/output data from CSV files.
    
    ILA convention:
    - u_pa: PA input (clean signal) -> TARGET for DPD
    - y_pa: PA output (distorted signal) -> INPUT to DPD
    """
    input_file = data_dir / f'{split}_input.csv'
    output_file = data_dir / f'{split}_output.csv'
    
    if not input_file.exists() or not output_file.exists():
        raise FileNotFoundError(f"Data files not found in {data_dir}")
    
    # Load CSV files
    input_df = pd.read_csv(input_file)
    output_df = pd.read_csv(output_file)
    
    # Convert to complex arrays
    u_pa = (input_df['I'].values + 1j * input_df['Q'].values).astype(np.complex64)
    y_pa = (output_df['I'].values + 1j * output_df['Q'].values).astype(np.complex64)
    
    # Normalize to -3 dBFS (0.7 linear)
    max_val = np.max(np.abs(u_pa))
    u_pa = u_pa / max_val * 0.7
    y_pa = y_pa / max_val * 0.7
    
    print(f"Loaded {len(u_pa):,} {split} samples")
    print(f"  PA input power:  {10*np.log10(np.mean(np.abs(u_pa)**2)):.2f} dBFS")
    print(f"  PA output power: {10*np.log10(np.mean(np.abs(y_pa)**2)):.2f} dBFS")
    
    return u_pa, y_pa

# Load data
data_dir = Path('data')
u_pa_train, y_pa_train = load_measured_data(data_dir, 'train')
u_pa_val, y_pa_val = load_measured_data(data_dir, 'val')
u_pa_test, y_pa_test = load_measured_data(data_dir, 'test')

In [ ]:
# Cell 5: Dataset Creation (matches train.py)
def create_dpd_dataset(u_pa: np.ndarray, y_pa: np.ndarray, memory_depth: int = 3) -> TensorDataset:
    """
    Create dataset for DPD training using Indirect Learning Architecture.
    
    ILA: DPD(y_PA) ≈ u_PA
    - Input to DPD: y_PA (distorted PA output)
    - Target: u_PA (clean PA input)
    
    PNTDNNGenerator handles phase-normalized feature extraction internally.
    We return raw IQ sequences [batch, M+1, 2].
    """
    num_samples = len(y_pa) - memory_depth
    
    # Input: raw IQ sequences from PA output
    inputs = np.zeros((num_samples, memory_depth + 1, 2), dtype=np.float32)
    # Target: IQ samples from PA input
    targets = np.zeros((num_samples, 2), dtype=np.float32)
    
    for i in range(num_samples):
        # Memory taps from PA output (time order: oldest to newest)
        for m in range(memory_depth + 1):
            idx = i + memory_depth - m
            inputs[i, m, 0] = y_pa[idx].real
            inputs[i, m, 1] = y_pa[idx].imag
        
        # Target is PA input at current time
        target_idx = i + memory_depth
        targets[i, 0] = u_pa[target_idx].real
        targets[i, 1] = u_pa[target_idx].imag
    
    return TensorDataset(torch.from_numpy(inputs), torch.from_numpy(targets))

# Create datasets
memory_depth = config['model']['generator']['memory_depth']
train_dataset = create_dpd_dataset(u_pa_train, y_pa_train, memory_depth)
val_dataset = create_dpd_dataset(u_pa_val, y_pa_val, memory_depth)
test_dataset = create_dpd_dataset(u_pa_test, y_pa_test, memory_depth)

# Create dataloaders
batch_size = config['training']['batch_size']
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"\nDataset sizes:")
print(f"  Training:   {len(train_dataset):,} samples, {len(train_loader):,} batches")
print(f"  Validation: {len(val_dataset):,} samples, {len(val_loader):,} batches")
print(f"  Test:       {len(test_dataset):,} samples, {len(test_loader):,} batches")

In [ ]:
# Cell 6: Model Creation (matches train.py)
# Generator: Phase-Normalized TDNN (24-dim input, 1,362 params)
generator = create_pn_tdnn_generator(memory_depth=config['model']['generator']['memory_depth'])
generator = generator.to(device)

# Discriminator: Conditional WGAN-GP (4-dim input)
discriminator = create_discriminator()
discriminator = discriminator.to(device)

# Verify architecture
g_params = sum(p.numel() for p in generator.parameters())
d_params = sum(p.numel() for p in discriminator.parameters())

print("Model Architecture Verification:")
print(f"  Generator:")
print(f"    Input dim: {generator.input_dim} (expected: 24)")
print(f"    Memory depth: M={generator.memory_depth} (expected: 3)")
print(f"    Hidden dims: {generator.hidden_dims}")
print(f"    Parameters: {g_params:,} (expected: 1,362)")
print(f"  Discriminator:")
print(f"    Parameters: {d_params:,}")

# Verify parameter count
assert g_params == 1362, f"Generator param count mismatch: {g_params} != 1362"
print("\n✓ Architecture verified!")

In [ ]:
# Cell 7: Optimizers and Schedulers (matches train.py)
opt_config = config['training']['optimizer']

g_optimizer = optim.Adam(
    generator.parameters(),
    lr=opt_config['lr_generator'],
    betas=tuple(opt_config['betas']),
    weight_decay=opt_config['weight_decay']
)

d_optimizer = optim.Adam(
    discriminator.parameters(),
    lr=opt_config['lr_discriminator'],
    betas=tuple(opt_config['betas']),
    weight_decay=opt_config['weight_decay']
)

# Cosine annealing scheduler
num_epochs = config['training']['epochs']
g_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    g_optimizer, T_max=num_epochs, eta_min=config['training']['scheduler']['min_lr']
)
d_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    d_optimizer, T_max=num_epochs, eta_min=config['training']['scheduler']['min_lr']
)

# Spectral loss
spectral_loss_fn = SpectralLoss(sample_rate=config['system']['sample_rate'])

print("Optimizers created:")
print(f"  Generator LR: {opt_config['lr_generator']}")
print(f"  Discriminator LR: {opt_config['lr_discriminator']}")
print(f"  Betas: {opt_config['betas']}")
print(f"  Scheduler: Cosine annealing to {config['training']['scheduler']['min_lr']}")

In [ ]:
# Cell 8: Training Step Function (matches train.py exactly)
def train_step(generator, discriminator, batch, g_optimizer, d_optimizer, 
               spectral_loss_fn, config, device):
    """
    Single training step using Indirect Learning Architecture (ILA).
    
    ILA:
    - Input: PA output (distorted signal y_PA)
    - Target: PA input (clean signal u_PA)
    - NO PA model in training loop!
    """
    train_config = config['training']
    loss_config = train_config['loss']
    n_critic = train_config['n_critic']
    gp_weight = train_config['gp_weight']
    
    # Unpack batch
    input_seq, target = batch  # [B, M+1, 2], [B, 2]
    input_seq = input_seq.to(device)
    target = target.to(device)
    
    losses = {}
    
    # ===================
    # Train Discriminator (n_critic times)
    # ===================
    for _ in range(n_critic):
        d_optimizer.zero_grad()
        
        with torch.no_grad():
            dpd_output = generator(input_seq)  # [B, 2]
        
        # Condition: current PA output sample
        condition = input_seq[:, -1, :]  # [B, 2]
        
        # Critic scores
        d_real = discriminator(target, condition)
        d_fake = discriminator(dpd_output.detach(), condition)
        
        # Wasserstein loss
        d_loss = d_fake.mean() - d_real.mean()
        
        # Gradient penalty
        batch_size = target.size(0)
        alpha = torch.rand(batch_size, 1, device=device)
        interpolates = alpha * target + (1 - alpha) * dpd_output.detach()
        interpolates.requires_grad_(True)
        
        d_interp = discriminator(interpolates, condition)
        gradients = torch.autograd.grad(
            outputs=d_interp,
            inputs=interpolates,
            grad_outputs=torch.ones_like(d_interp),
            create_graph=True,
            retain_graph=True,
        )[0]
        
        gradients_norm = gradients.view(batch_size, -1).norm(2, dim=1)
        gp = ((gradients_norm - 1) ** 2).mean()
        
        d_total = d_loss + gp_weight * gp
        d_total.backward()
        d_optimizer.step()
    
    losses['d_wasserstein'] = d_loss.item()
    losses['d_gp'] = gp.item()
    losses['d_total'] = d_total.item()
    
    # ===================
    # Train Generator
    # ===================
    g_optimizer.zero_grad()
    
    dpd_output = generator(input_seq)  # [B, 2]
    
    # Adversarial loss
    d_fake = discriminator(dpd_output, condition)
    g_adv_loss = -d_fake.mean()
    
    # Reconstruction loss (L1)
    recon_loss = nn.functional.l1_loss(dpd_output, target)
    
    # Spectral loss
    spectral, spectral_components = spectral_loss_fn(dpd_output, target, return_components=True)
    
    # Combined loss
    g_total = (
        loss_config['adversarial'] * g_adv_loss +
        loss_config['reconstruction_l1'] * recon_loss +
        loss_config['spectral'] * spectral
    )
    
    g_total.backward()
    g_optimizer.step()
    
    losses['g_adv'] = g_adv_loss.item()
    losses['g_recon'] = recon_loss.item()
    losses['g_spectral'] = spectral.item()
    losses['g_total'] = g_total.item()
    losses.update({f'g_{k}': v.item() for k, v in spectral_components.items()})
    
    return losses

print("Training step function defined (ILA - no PA in loop)")

In [ ]:
# Cell 9: Validation Function
def validate(generator, val_loader, spectral_loss_fn, device):
    """Validate model on validation set."""
    generator.eval()
    
    all_evm = []
    all_nmse = []
    all_recon = []
    
    with torch.no_grad():
        for input_seq, target in val_loader:
            input_seq = input_seq.to(device)
            target = target.to(device)
            
            dpd_output = generator(input_seq)
            metrics = spectral_loss_fn.compute_metrics(dpd_output, target)
            
            all_evm.append(metrics['evm_db'])
            all_nmse.append(metrics['nmse_db'])
            all_recon.append(metrics['l1_error'])
    
    generator.train()
    
    return {
        'val_evm_db': np.mean(all_evm),
        'val_nmse_db': np.mean(all_nmse),
        'val_l1': np.mean(all_recon)
    }

print("Validation function defined")

In [ ]:
# Cell 10: Training Loop with QAT Transition
# Training history
history = {
    'g_total': [], 'd_total': [], 'g_adv': [], 'g_recon': [], 'g_spectral': [],
    'val_evm_db': [], 'val_nmse_db': [], 'val_l1': [], 'lr': []
}

# Best model tracking
best_evm = float('inf')
best_state = None

# QAT configuration
qat_start_epoch = config['qat']['start_epoch']
qat_enabled = False

print(f"Starting training for {num_epochs} epochs...")
print(f"QAT will be enabled at epoch {qat_start_epoch}")
print("="*60)

for epoch in range(num_epochs):
    generator.train()
    discriminator.train()
    
    # ==================
    # QAT Transition
    # ==================
    if epoch == qat_start_epoch and config['qat']['enabled'] and not qat_enabled:
        generator.enable_qat()
        qat_enabled = True
        print(f"\n{'='*60}")
        print(f"QAT ENABLED at epoch {epoch}")
        print(f"  Weight quantization: Q1.15 ({config['qat']['weight_bits']} bits)")
        print(f"  Activation quantization: Q8.8 ({config['qat']['activation_bits']} bits)")
        print(f"{'='*60}\n")
    
    # ==================
    # Training epoch
    # ==================
    epoch_losses = {}
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
    
    for batch in pbar:
        losses = train_step(
            generator, discriminator, batch,
            g_optimizer, d_optimizer, spectral_loss_fn,
            config, device
        )
        
        for k, v in losses.items():
            epoch_losses[k] = epoch_losses.get(k, 0) + v
        
        pbar.set_postfix({
            'g': f"{losses['g_total']:.4f}",
            'd': f"{losses['d_total']:.4f}",
            'QAT': 'ON' if qat_enabled else 'OFF'
        })
    
    # Average losses
    for k in epoch_losses:
        epoch_losses[k] /= len(train_loader)
    
    # ==================
    # Validation
    # ==================
    val_metrics = validate(generator, val_loader, spectral_loss_fn, device)
    
    # Update history
    for k in ['g_total', 'd_total', 'g_adv', 'g_recon', 'g_spectral']:
        history[k].append(epoch_losses.get(k, 0))
    for k in ['val_evm_db', 'val_nmse_db', 'val_l1']:
        history[k].append(val_metrics[k])
    history['lr'].append(g_optimizer.param_groups[0]['lr'])
    
    # Update schedulers
    g_scheduler.step()
    d_scheduler.step()
    
    # Save best model
    if val_metrics['val_evm_db'] < best_evm:
        best_evm = val_metrics['val_evm_db']
        best_state = {
            'epoch': epoch,
            'generator_state_dict': generator.state_dict(),
            'discriminator_state_dict': discriminator.state_dict(),
            'best_evm': best_evm,
            'qat_enabled': qat_enabled,
        }
    
    # Print epoch summary (every 10 epochs)
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"\nEpoch {epoch+1}/{num_epochs} {'[QAT]' if qat_enabled else ''}")
        print(f"  G Loss: {epoch_losses['g_total']:.4f} | D Loss: {epoch_losses['d_total']:.4f}")
        print(f"  Val EVM: {val_metrics['val_evm_db']:.2f} dB | NMSE: {val_metrics['val_nmse_db']:.2f} dB")
        print(f"  Best EVM: {best_evm:.2f} dB | LR: {g_optimizer.param_groups[0]['lr']:.2e}")

print(f"\n{'='*60}")
print(f"Training complete!")
print(f"Best EVM: {best_evm:.2f} dB at epoch {best_state['epoch']+1}")

In [ ]:
# Cell 11: Plot Training Curves
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Generator loss
axes[0, 0].plot(history['g_total'], label='Total')
axes[0, 0].plot(history['g_adv'], label='Adversarial', alpha=0.7)
axes[0, 0].plot(history['g_recon'], label='Reconstruction', alpha=0.7)
axes[0, 0].axvline(x=qat_start_epoch, color='r', linestyle='--', label='QAT Start')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Generator Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Discriminator loss
axes[0, 1].plot(history['d_total'])
axes[0, 1].axvline(x=qat_start_epoch, color='r', linestyle='--', label='QAT Start')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].set_title('Discriminator Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Spectral loss
axes[0, 2].plot(history['g_spectral'])
axes[0, 2].axvline(x=qat_start_epoch, color='r', linestyle='--', label='QAT Start')
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylabel('Spectral Loss')
axes[0, 2].set_title('Spectral Loss (ACPR/EVM)')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# EVM
axes[1, 0].plot(history['val_evm_db'])
axes[1, 0].axhline(y=-45, color='g', linestyle='--', label='Target (-45 dB)')
axes[1, 0].axvline(x=qat_start_epoch, color='r', linestyle='--', label='QAT Start')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('EVM (dB)')
axes[1, 0].set_title('Validation EVM')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# NMSE
axes[1, 1].plot(history['val_nmse_db'])
axes[1, 1].axhline(y=-42, color='g', linestyle='--', label='Target (-42 dB)')
axes[1, 1].axvline(x=qat_start_epoch, color='r', linestyle='--', label='QAT Start')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('NMSE (dB)')
axes[1, 1].set_title('Validation NMSE')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Learning rate
axes[1, 2].plot(history['lr'])
axes[1, 2].axvline(x=qat_start_epoch, color='r', linestyle='--', label='QAT Start')
axes[1, 2].set_xlabel('Epoch')
axes[1, 2].set_ylabel('Learning Rate')
axes[1, 2].set_title('Learning Rate Schedule')
axes[1, 2].set_yscale('log')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/results/training_curves_v2.png', dpi=150)
plt.show()

In [ ]:
# Cell 12: Save Best Model
# Load best model
generator.load_state_dict(best_state['generator_state_dict'])

# Save checkpoint
checkpoint_path = Path('checkpoints/pn_tdnn_best.pth')
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

torch.save({
    'epoch': best_state['epoch'],
    'generator_state_dict': best_state['generator_state_dict'],
    'discriminator_state_dict': best_state['discriminator_state_dict'],
    'best_evm': best_state['best_evm'],
    'qat_enabled': best_state['qat_enabled'],
    'config': config,
}, checkpoint_path)

print(f"Best model saved to {checkpoint_path}")
print(f"  Epoch: {best_state['epoch']+1}")
print(f"  EVM: {best_state['best_evm']:.2f} dB")
print(f"  QAT: {'Enabled' if best_state['qat_enabled'] else 'Disabled'}")

In [ ]:
# Cell 13: Final Evaluation on Test Set
generator.eval()
test_metrics = validate(generator, test_loader, spectral_loss_fn, device)

print("="*60)
print("FINAL TEST SET RESULTS")
print("="*60)
print(f"  EVM:  {test_metrics['val_evm_db']:.2f} dB (target: < -45 dB)")
print(f"  NMSE: {test_metrics['val_nmse_db']:.2f} dB (target: < -42 dB)")
print(f"  L1:   {test_metrics['val_l1']:.6f}")
print("="*60)

# Check if targets met
evm_pass = test_metrics['val_evm_db'] < -45
nmse_pass = test_metrics['val_nmse_db'] < -42
print(f"\nEVM target:  {'✓ PASS' if evm_pass else '✗ FAIL'}")
print(f"NMSE target: {'✓ PASS' if nmse_pass else '✗ FAIL'}")

In [ ]:
# Cell 14: Export Weights for FPGA
# Export Q1.15 quantized weights
weights_q115 = generator.export_weights_q115()

# Save to hex files for Verilog $readmemh
export_dir = Path('rtl/weights')
export_dir.mkdir(parents=True, exist_ok=True)

for name, weights in weights_q115.items():
    # Convert to hex
    hex_values = []
    for val in weights.flatten():
        # Convert signed int16 to unsigned for hex representation
        if val < 0:
            val = val + 65536  # 2's complement
        hex_values.append(f"{val:04X}")
    
    # Save to file
    clean_name = name.replace('.', '_')
    hex_path = export_dir / f"{clean_name}.hex"
    with open(hex_path, 'w') as f:
        f.write('\n'.join(hex_values))
    
    print(f"Exported {name}: {weights.shape} -> {hex_path}")

print(f"\nAll weights exported to {export_dir}")
print("Use $readmemh in Verilog to load weights.")

## Summary

This notebook implements the production training flow for PN-TDNN-DPD:

1. **Architecture**: Phase-Normalized TDNN (24-dim, M=3, 1,362 params)
2. **Training**: CWGAN-GP + Spectral Loss + ILA (no PA in loop)
3. **QAT**: Enabled automatically at epoch 300
4. **Export**: Q1.15 weights ready for FPGA

### Code Block Summary

| Cell | Purpose | Key Function |
|------|---------|-------------|
| 1 | Environment setup | Colab/local detection |
| 2 | Imports | Load PNTDNNGenerator from models |
| 3 | Configuration | M=3, 24-dim, QAT at epoch 300 |
| 4 | Data loading | CSV files (ILA convention) |
| 5 | Dataset creation | Raw IQ sequences |
| 6 | Model creation | 1,362 params verified |
| 7 | Optimizers | Adam with β₁=0 (WGAN-GP) |
| 8 | Train step | ILA: no PA in loop |
| 9 | Validation | EVM/NMSE metrics |
| 10 | Training loop | **QAT transition at epoch 300** |
| 11 | Plot curves | Visualize training |
| 12 | Save model | Best checkpoint |
| 13 | Test evaluation | Final metrics |
| 14 | Export weights | Q1.15 hex files |